# A100 데이터 전처리 비교 실험

모델: V6와 동일한 동결 MERT-95M + 768→1 선형 분류층. 음성·존재 확률은 기존 V6 캐시를 유지합니다. 모델 구조를 바꾸지 않고 출처 가중치, 파일당 구간 기여량, 4초 crop의 효과를 순서대로 비교합니다.

**시간:** GPU 시작부터 최대 4시간. 3시간 30분에 연산을 종료하고 다운로드·Pause 시간을 확보합니다. 완료 즉시 결과를 내려받고 Pause합니다.

**평가:** 기존 dev 210개와 PCM 중복 제거 190개를 병기합니다. 개발 데이터는 이미 사용한 자료이며 리더보드 성능이나 미사용 holdout을 대신하지 않습니다. 성능이 나쁜 파일의 라벨을 바꾸거나 삭제하지 않습니다.


## 1. 실행 설정

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, json, os, sys, time
import numpy as np
import torch
PROJECT = Path('/root/V6_L40S')
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(Path.cwd()))
from v6_data import write_json, sha256
from v6_core import Heads
from v7_ablation import Recipe, run_candidate, evaluate
from v7_features import read_existing, fresh_features
RESUMED = float(os.environ['V7_RESUMED_EPOCH'])
DEADLINE = RESUMED + 3.5 * 3600
OUTPUT = Path.cwd() / 'results'
OUTPUT.mkdir(exist_ok=True)
torch.set_num_threads(4)
assert time.time() < DEADLINE, '시간 한도 경과'
assert torch.cuda.is_available(), 'A100 연결 필요'
print({'gpu': torch.cuda.get_device_name(), 'torch': torch.__version__,
       'compute_deadline_utc': datetime.fromtimestamp(DEADLINE, timezone.utc).isoformat(),
       'workspace_pause_deadline_utc': datetime.fromtimestamp(RESUMED + 14400, timezone.utc).isoformat()})


## 2. 입력·분할·캐시 검증
이전 실행의 입력 파일 해시·캐시 식별자를 확인합니다. GPU가 달라졌다는 이유로 이미 계산된 특징을 다시 만들지 않습니다.

In [ ]:
split, records, previous = read_existing(PROJECT)
expected = json.loads(Path('input_manifest.json').read_text())
assert sha256(PROJECT / 'v6_run/split.json') == expected['v6_split_sha256']
unique = list({r['pcm_sha256']: r for r in reversed(split['dev'])}.values())
assert len(split['dev']) == 210 and len(unique) == 190
assert {r['ID'] for r in unique} == {r['ID'] for r in csv.DictReader(Path('dev_unique_ids.csv').open(encoding='utf-8-sig'))}
write_json(OUTPUT / 'input_verification.json', {'v6_metadata': previous, 'records': len(records),
    'dev_files': len(split['dev']), 'unique_dev_files': len(unique), 'input_manifest': expected})
print({'verified_cache_records': len(records), 'unique_dev': len(unique)})


## 3. 기존 V6 점수 재현
저장된 가중치와 캐시가 이전 개발 예측을 재현하는지 확인합니다.

In [ ]:
head = Heads.load(PROJECT / 'v6_run/heads.npz')
predictions, original_metrics = evaluate(head, split['dev'], records)
old_predictions = {r['ID']: r for r in json.loads((PROJECT / 'v6_run/dev_predictions.json').read_text())}
from v6_core import COLUMNS
reproduction_delta = max(abs(row[k] - old_predictions[row['ID']][k]) for row in predictions for k in COLUMNS)
reference_metrics = json.loads((PROJECT / 'v6_run/dev_metrics.json').read_text())['metrics']
write_json(OUTPUT / 'saved_head_reproduction.json', {'maximum_probability_difference': reproduction_delta,
    'recomputed_metrics': original_metrics['metrics'], 'reference_metrics': reference_metrics,
    'absolute_probability_tolerance': 1e-5, 'reason': 'Float32 BLAS reductions may differ across CPU hosts.'})
assert reproduction_delta < 1e-5
for key, value in reference_metrics.items():
    if value is not None:
        np.testing.assert_allclose(original_metrics['metrics'][key], value, atol=1e-12, rtol=0)
_, dedup_metrics = evaluate(head, unique, records)
reports = {'V6_saved': {'deduplicated': dedup_metrics, 'full': original_metrics}}
write_json(OUTPUT / 'comparison.json', reports)
print(dedup_metrics['metrics'])


## 4. 캐시를 그대로 쓰는 전처리 비교

- C0: V6와 같은 클래스 균형, 전체 구간 MIL.
- C1: FMA 45%, NSynth 5%, 가짜 50%. 가짜는 생성기별 균등 가중치.
- C2: NSynth 제외, FMA 50%, 가짜 50%.
- C3: C1 가중치 + 파일당 동일한 정규화 기여량 + epoch마다 파일당 구간 하나 선택.

AdamW lr=0.003, weight_decay=0.01, 300 epoch, 선형층 초기값 0을 고정합니다. 결합기 학습 자료는 그대로 유지하고 각 후보의 음악 점수로 다시 학습합니다. 기존 캐시의 실제 구간 길이는 64,600 samples(4.0375초)이며, 다음 단계에서 정확히 4초로 비교합니다.


In [ ]:
recipes = [Recipe('C0_class_balance', None), Recipe('C1_nsynth_5pct', .05),
           Recipe('C2_no_nsynth', 0.), Recipe('C3_equal_windows', .05, True)]
for recipe in recipes:
    write_json(OUTPUT / 'progress.json', {'stage': 'training', 'candidate': recipe.name, 'utc': time.time()})
    reports[recipe.name] = run_candidate(recipe, (split, records, OUTPUT), DEADLINE)
    write_json(OUTPUT / 'comparison.json', reports)
    print(recipe.name, reports[recipe.name]['deduplicated']['metrics'])
control = json.loads((OUTPUT / 'C0_class_balance/predictions.json').read_text())
max_delta = max(abs(r[k] - old_predictions[r['ID']][k]) for r in control for k in COLUMNS)
write_json(OUTPUT / 'control_reproduction.json', {'maximum_probability_difference': max_delta,
    'note': 'CPU numerical differences are recorded; same loss and optimizer, no hyperparameter search.'})
assert max_delta < .002, '기존 학습 재현 차이를 먼저 확인해야 합니다.'


## 5. 정확히 4초인 음악 구간 재계산

원본 파일을 유지한 채 HTDemucs 음악 성분에서 64,000 samples를 추출합니다. 단독 학습 음원은 파일별 최대 4개의 재현 가능한 위치를 미리 계산하고, 각 epoch에서 하나를 선택합니다. 이는 매 epoch 새 위치를 추론하는 방식이 아니며 GPU 비용을 제한하기 위한 고정 crop pool입니다. 짧은 음원은 반복하지 않고 필요한 경우 0으로 패딩합니다.

개발·결합기 자료는 기존 시간 위치와 구간 수를 유지해 음성·존재 확률과 정렬합니다. 무음 비율은 기록하며 이번 비교에서 자동 삭제 기준으로 사용하지 않습니다. 개발 음원의 DF-Arena 음성·음악 점수도 다시 계산해 공식 베이스라인과 비교합니다. 베이스라인의 PANNs 존재 확률은 검증된 기존 캐시를 재사용합니다.


In [ ]:
fresh, baseline = fresh_features((split, records, OUTPUT), PROJECT / 'model/assets', DEADLINE)
from graph_scoring import official_metrics
baseline_by_id = {r['ID']: r for r in baseline}
reports['Official_baseline'] = {'full': official_metrics(split['dev'], baseline),
    'deduplicated': official_metrics(unique, [baseline_by_id[r['ID']] for r in unique])}
write_json(OUTPUT / 'comparison.json', reports)
print('Official baseline', reports['Official_baseline']['deduplicated']['metrics'])


## 6. 4초·동일 구간 기여량 조건 학습
C4는 NSynth 5%, C5는 NSynth 제외입니다. 구간 선택 변동을 보기 위해 seed 41·42·43을 모두 기록하며 가장 좋은 seed만 골라 보고하지 않습니다.

In [ ]:
for seed in (41, 42, 43):
    for name, mass in [('C4_four_seconds', .05), ('C5_four_seconds_no_nsynth', 0.)]:
        recipe = Recipe(f'{name}_seed{seed}', mass, True, seed)
        write_json(OUTPUT / 'progress.json', {'stage': 'training', 'candidate': recipe.name, 'utc': time.time()})
        reports[recipe.name] = run_candidate(recipe, (split, fresh, OUTPUT), DEADLINE)
        write_json(OUTPUT / 'comparison.json', reports)
        print(recipe.name, reports[recipe.name]['deduplicated']['metrics'])


## 7. 결과 비교
EER는 낮을수록 좋습니다. 모든 후보는 동일한 개발 ID를 사용합니다. 음성 presence 라벨이 충분하지 않아 종합 점수는 계산할 수 없습니다. 혼합·부분 위조의 생성기와 처리 조건도 얽혀 있으므로 부분집합 점수를 인과관계로 해석하지 않습니다.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image
rows = []
for name, report in reports.items():
    m = report['deduplicated']['metrics']
    rows.append({'candidate': name, 'file_eer': m['file_eer'], 'music_eer': m['music_eer']})
with (OUTPUT / 'comparison.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]), lineterminator='\n')
    writer.writeheader(); writer.writerows(rows)
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(rows))
ax.barh(x - .18, [r['file_eer'] for r in rows], height=.35, label='File EER')
ax.barh(x + .18, [r['music_eer'] for r in rows], height=.35, label='Music EER')
ax.set_yticks(x, [r['candidate'] for r in rows]); ax.invert_yaxis()
ax.set_xlabel('EER (lower is better)'); ax.set_xlim(0, 1)
ax.legend(); fig.tight_layout(); fig.savefig(OUTPUT / 'comparison.png', dpi=150)
display(Image(filename=str(OUTPUT / 'comparison.png')))
for row in rows: print(row)
write_json(OUTPUT / 'progress.json', {'stage': 'complete', 'utc': time.time(), 'candidates': len(rows)})
print('학습·평가 완료. 실행 노트북과 결과를 다운로드한 뒤 VESSL workspace를 Pause합니다.')
